### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="qsar_fish_toxicity",
    dataset_year="2015",
    domain_str="biology & life sciences",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5JG7B",
    download_description="""
We download the data from the UCI repository and unzip it to a predefined folder.

mkdir -p local-data-warehouse/qsar_fish_toxicity/ && wget -P local-data-warehouse/qsar_fish_toxicity/ https://archive.ics.uci.edu/static/public/504/qsar+fish+toxicity.zip && unzip local-data-warehouse/qsar_fish_toxicity/qsar+fish+toxicity.zip -d local-data-warehouse/qsar_fish_toxicity/
""",
    # References
    academic_reference_bibtex="""@article{cassotti2015similarity,
  title={A similarity-based QSAR model for predicting acute toxicity towards the fathead minnow (Pimephales promelas)},
  author={Cassotti, Matteo and Ballabio, Davide and Todeschini, Roberto and Consonni, Viviana},
  journal={SAR and QSAR in Environmental Research},
  volume={26},
  number={3},
  pages={217--243},
  year={2015},
  publisher={Taylor \& Francis}
}
""",
    academic_reference_bibtex_key="cassotti2015similarity",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
- We assigned descriptive column names following the original dataset documentation.
- Anomaly: the data contains a lot of duplicates (15%) when ignoring the target feature.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="LC50",
    problem_type="regression",
    objective_metric_name="rmse",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_csv(f"{dataset_mold.path}/qsar_fish_toxicity.csv", sep=";", header=None)

target_feature = "LC50"
df.columns = [
    "CIC0",
    "SM1_Dz(Z)",
    "GATS1i",
    "NdsCH",
    "NdssC",
    "MLOGP",
    target_feature,
]

# Original data is ordered and thus we have shift that vanishes after shuffling
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 908
Columns: 7
Use sampling: False (sample size: 908)
Get row duplicates (staged, merged)...
Using top-6 columns for initial filtering: ['MLOGP', 'GATS1i', 'CIC0', 'SM1_Dz(Z)', 'NdssC', 'NdsCH']
Rows remaining as candidates after top-6 filter: 217 (of 908)

#### Duplicate Report
Total duplicate rows: 1 (0.11% of dataset)
Duplicate rows ignoring target: 133 (14.65% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,CIC0,SM1_Dz(Z),GATS1i,NdsCH,NdssC,MLOGP,LC50
0,1.783,0.629,0.750,1,0,0.736,3.460
1,4.181,0.570,1.383,0,0,4.379,6.213
2,2.137,0.223,1.179,0,0,0.655,3.779
3,2.748,0.223,1.705,0,0,0.800,1.691
4,1.417,0.898,0.648,0,0,2.042,4.421


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,CIC0,float64,0.0,0.0,502.0,"2.126, 3.08, 2.377, 2.08, 2.508, 3.252, 2.834, 2.479, 3.179, 2.216"
1,SM1_Dz(Z),float64,0.0,0.0,186.0,"0.223, 0.134, 0.405, 0.331, 0.0, 0.693, 0.56, 0.496, 0.251, 0.83"
2,GATS1i,float64,0.0,0.0,557.0,"0.941, 0.938, 1.179, 0.954, 0.871, 1.189, 1.571, 1.6, 1.705, 1.288"
3,MLOGP,float64,0.0,0.0,559.0,"0.8, 1.701, 0.202, 1.064, 1.748, 2.604, 1.587, 1.442, 2.193, 1.859"
4,LC50,float64,0.0,0.0,827.0,"4.208, 3.513, 3.979, 3.926, 3.47, 3.66, 4.739, 3.92, 4.691, 3.84"
5,NdsCH,int64,0.0,0.0,5.0,"0, 1, 2, 4, 3"
6,NdssC,int64,0.0,0.0,7.0,"0, 1, 2, 3, 4, 6, 5"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
CIC0,908.0,2.898129,0.756088,0.667,5.926
SM1_Dz(Z),908.0,0.628468,0.428459,0.000,2.171
GATS1i,908.0,1.293591,0.394303,0.396,2.920
NdsCH,908.0,0.229075,0.605335,0.000,4.000
NdssC,908.0,0.485683,0.861279,0.000,6.000
MLOGP,908.0,2.109285,1.433181,-2.884,6.515
LC50,908.0,4.064431,1.455698,0.053,9.612


In [7]:
# Categorical Feature Statistics
cat_stats

'No categorical/object features to summarize.'

In [8]:
# Target Distribution
target_df

,y_missing_count,non_positive_pct,skew_y,skew_log,var_y,var_log,log_used,aic_exponential,aic_lognormal,dist_hint
0,0,0.0,0.252,-2.464,2.119,0.222,log,4364.7,777554.5,exponential


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=10, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to qsar_fish_toxicity/019d7369-62fc-7ea1-9c58-8ac89144f820
019d7369-62fc-7ea1-9c58-8ac89144f820
4f9aed32b921567469a36e51373f11e8b37193c773946c6501fa1dd6bcfad69d
